<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/Llama3_1_%ED%8A%B9%ED%97%88_%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC_%EB%B6%84%EB%A5%98_%ED%94%84%EB%A1%AC%ED%94%84%ED%8A%B8_%ED%8A%9C%EB%8B%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama 3.1을 이용해서 특허 카테고리 자동 분류하기 - 프롬프트 튜닝을 통한 성능 개선
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## Reference : https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
## [특허 분야 자동분류 데이터] 데이터 : https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&dataSetSn=547

In [ ]:
!nvidia-smi

# 라이브러리 설치

In [ ]:
import torch

# 1. GPU 확인 (잘 잡혀있는지 체크)
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("GPU가 없습니다. 상단 메뉴 [런타임] -> [런타임 유형 변경]에서 T4 GPU를 선택하세요.")

print("\nInstalling Libraries for Llama-3.1...")

# 2. 필수 라이브러리 설치
!pip install -U "transformers>=4.43.0" "accelerate>=0.26.0" "bitsandbytes>=0.42.0" "huggingface_hub"

print("\nSetup Completed.")

In [ ]:
import json
import pandas as pd

## [특허 분야 자동분류 데이터] 데이터 불러오기

In [ ]:
!unzip Patent_AutoMatic_Regression.zip

## 원천 데이터 불러오기

In [ ]:
# 파일 경로
file_path = '/content/원천데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01110_곡물및기타식량작물재배업.json'

# 파일 열기 및 JSON 읽기
with open(file_path, 'r', encoding='utf-8') as file:
    raw_data = json.load(file)
raw_data = raw_data['dataset']

In [ ]:
raw_data

In [ ]:
len(raw_data)

In [ ]:
raw_data[0]

In [ ]:
raw_data[0].keys()

In [ ]:
def extract_fields(data):
    # 필요한 필드를 추출
    invention_title = data.get('invention_title', '')
    abstract = data.get('abstract', '')
    claims = data.get('claims', '')

    # 추출한 내용을 하나의 문자열로 연결
    result = f"invention_title: {invention_title} abstract: {abstract} claims: {claims}"

    return result

In [ ]:
data_0 = extract_fields(raw_data[0])
data_0

## 라벨링 데이터 불러오기

In [ ]:
# 파일 경로
file_path = '/content/라벨링데이터/A_농업_임업및어업_01_03/01_농업/011_작물재배업/01110_곡물및기타식량작물재배업.json'

# 파일 열기 및 JSON 읽기
with open(file_path, 'r', encoding='utf-8') as file:
    label_data = json.load(file)
label_data = label_data['dataset']

In [ ]:
label_data

In [ ]:
len(label_data)

In [ ]:
label_data[0]

In [ ]:
label_data[0].keys()

In [ ]:
label_data[0]['Ltext']

# Llama 3.1 모델 불러오기

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# 1. Hugging Face Token 설정
os.environ['HF_TOKEN'] = "Input Your Token"


# 2. 모델 설정 및 로드
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# [수정 1] T4 GPU 메모리(16GB) 초과 방지를 위한 4-bit 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16 # T4는 float16 권장
)

print(f" Loading Model: {model_id}...")

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Llama 모델 에러 방지용 설정 (Padding Token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# [수정 2] 모델 로드 (bfloat16 제거 -> float16 & 4bit 적용)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config, # 4-bit 양자화 적용
    device_map="auto",
    torch_dtype=torch.float16       # T4 호환성 맞춤
)

print(f"Model Loaded Successfully on {model.device}")

In [ ]:
def generate_response(system_message, user_message, tokenizer, model):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=False,   # 항상 가장 확률이 높은 값으로 예측
        temperature=0.6,
        top_p=0.9
    )
    response = outputs[0][input_ids.shape[-1]:]

    return tokenizer.decode(response, skip_special_tokens=True)

In [ ]:
print(model.device)

# 특허 카테고리 분류 테스트

In [ ]:
# 특허_카테고리_list = ['임업',
#  '어업',
#  '농업',
#  '자동차 및 트레일러 제조업',
#  '인쇄 및 기록매체 복제업',
#  '펄프, 종이 및 종이제품 제조업',
#  '비금속 광물제품 제조업',
#  '고무 및 플라스틱제품 제조업',
#  '코크스, 연탄 및 석유정제품 제조업',
#  '음료 제조업',
#  '의복, 의복 액세서리 및 모피제품 제조업',
#  '가구 제조업',
#  '기타 운송장비 제조업',
#  '전기장비 제조업',
#  '의료용 물질 및 의약품 제조업',
#  '기타 제품 제조업',
#  '산업용 기계 및 장비 수리업',
#  '화학 물질 및 화학제품 제조업; 의약품 제외',
#  '목재 및 나무제품 제조업; 가구 제외',
#  '전자 부품, 컴퓨터, 영상, 음향 및 통신장비 제조업',
#  '섬유제품 제조업; 의복 제외',
#  '담배 제조업',
#  '의료, 정밀, 광학 기기 및 시계 제조업',
#  '금속 가공제품 제조업; 기계 및 가구 제외',
#  '기타 기계 및 장비 제조업',
#  '가죽, 가방 및 신발 제조업',
#  '1차 금속 제조업',
#  '식료품 제조업',
#  '금속 광업',
#  '광업 지원 서비스업',
#  '석탄, 원유 및 천연가스 광업',
#  '비금속광물 광업; 연료용 제외']

In [ ]:
특허_카테고리_list = ['임업',
 '어업',
 '농업']

In [ ]:
len(특허_카테고리_list)

In [ ]:
#System Prompt Prompt Tuning 전 기본 프롬프트
system_prompt = f"너는 특허 카테고리를 분류하는 전문가야. \
아래 내용을 다음 특허 카테고리 중 하나로 분류해줘. 가능한 특허 카테고리 : {특허_카테고리_list}. \
최종 출력 결과는 다른말은 하지말고 분류한 카테고리만 출력해줘."
system_prompt

In [ ]:
data_0

In [ ]:
#Prompt-Tuning 전에 대한 답
llama3_1_inference_result = generate_response(system_message=system_prompt,
                              user_message=data_0,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_1_inference_result)
# 정답 : 농업

In [ ]:
raw_data[1]

In [ ]:
data_1 = extract_fields(raw_data[1])
data_1

In [ ]:
label_data[1]

In [ ]:
#Prompt-Tuning 전에 프롬프트에 대한 답변
llama3_1_inference_result = generate_response(system_message=system_prompt,
                              user_message=data_1,
                              tokenizer=tokenizer,
                              model=model)
print(llama3_1_inference_result)
# 정답 : 농업

# 전체 데이터 불러오기

In [ ]:
def list_all_files_in_directory(directory_path):
    # 파일 경로를 저장할 리스트 초기화
    all_files = []

    # directory_path 하위 폴더의 모든 파일들의 경로를 리스트에 저장
    for root, dirs, files in os.walk(directory_path):
        for file in files:
            # 파일 경로를 리스트에 추가
            all_files.append(os.path.join(root, file))

    # 전체 파일 경로 리스트 반환
    return all_files

In [ ]:
directory_path = '/content/원천데이터'
raw_file_list = list_all_files_in_directory(directory_path)
raw_file_list

In [ ]:
len(raw_file_list)

In [ ]:
def load_json_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)

    return data.get('dataset')

In [ ]:
dataframes = []
for idx, file in enumerate(raw_file_list):
    print(idx, file)
    data = load_json_dataset(file)
    df = pd.DataFrame(data)
    dataframes.append(df)  # 리스트에 DataFrame 추가

raw_df = pd.concat(dataframes, ignore_index=True)

In [ ]:
raw_df

In [ ]:
raw_df.shape

In [ ]:
directory_path = '/content/라벨링데이터'
label_file_list = list_all_files_in_directory(directory_path)
label_file_list

In [ ]:
len(label_file_list)

In [ ]:
dataframes = []
for idx, file in enumerate(label_file_list):
    print(idx, file)
    data = load_json_dataset(file)
    df = pd.DataFrame(data)
    dataframes.append(df)  # 리스트에 DataFrame 추가

label_df = pd.concat(dataframes, ignore_index=True)

In [ ]:
label_df

In [ ]:
label_df.shape

In [ ]:
특허_카테고리_list = label_df['Ltext'].unique().tolist()
특허_카테고리_list

In [ ]:
len(특허_카테고리_list)

# 중복된 값 제거

In [ ]:
# raw_df에서 중복된 application_number 확인
print(raw_df['application_number'].duplicated().sum())

# label_df에서 중복된 application_number 확인
print(label_df['application_number'].duplicated().sum())

In [ ]:
# 중복된 application_number 찾기
duplicate_application_numbers = raw_df[raw_df['application_number'].duplicated()]['application_number'].unique()
duplicate_application_numbers

In [ ]:
duplicate_rows = raw_df[raw_df['application_number'].isin(['1020190179476'])]

In [ ]:
duplicate_rows

In [ ]:
# raw_df에서 application_number 기준으로 중복된 행 제거
raw_df_unique = raw_df.drop_duplicates(subset='application_number', keep='first')

# label_df에서 application_number 기준으로 중복된 행 제거
label_df_unique = label_df.drop_duplicates(subset='application_number', keep='first')

In [ ]:
raw_df_unique.shape

In [ ]:
label_df_unique.shape

In [ ]:
merged_df = pd.merge(raw_df_unique, label_df_unique, on='application_number', how='inner')
merged_df

In [ ]:
merged_df.shape

In [ ]:
def extract_fields(row):
    # 각 행(row)에서 필요한 필드를 추출
    invention_title = row.get('invention_title', '')
    abstract = row.get('abstract', '')
    claims = row.get('claims', '')

    # 추출한 내용을 하나의 문자열로 연결
    result = f"invention_title: {invention_title} abstract: {abstract} claims: {claims}"

    return result

# apply 함수를 사용하여 merged_df의 각 행에 대해 extract_fields 함수를 적용
merged_df['combined_string'] = merged_df.apply(extract_fields, axis=1)

# 결과 출력 (예시로 첫 5개의 결과 출력)
merged_df[['invention_title', 'abstract', 'claims', 'combined_string']].head()

In [ ]:
merged_df

In [ ]:
test_data_df = merged_df[['application_number', 'combined_string', 'Ltext']]
test_data_df

In [ ]:
# '농업', '임업', '어업' 데이터만 추출
filtered_df = test_data_df[test_data_df['Ltext'].isin(['농업', '임업', '어업'])]
filtered_df

In [ ]:
import csv

# 1. 기존 CSV 파일을 읽어서 application_number 목록을 추출
try:
    existing_df = pd.read_csv('test_output.csv', encoding='utf-8')
    existing_app_numbers = set(existing_df['application_number'].astype(str))
except FileNotFoundError:
    # 파일이 없을 경우 빈 set으로 초기화
    existing_app_numbers = set()

In [ ]:
total_num = len(filtered_df)

# 2. 새 데이터를 추가할 CSV 파일을 연다
with open('test_output.csv', mode='a', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, quoting=csv.QUOTE_MINIMAL)

    # 3. 데이터프레임 순회하면서 application_number가 중복되지 않으면 추가
    for i, (index, row) in enumerate(filtered_df.iterrows()):
        app_number = str(row['application_number'])  # 문자열로 변환하여 비교
        if app_number not in existing_app_numbers:
            llama3_1_inference_result = generate_response(system_message=system_prompt,
                                          user_message=row['combined_string'],
                                          tokenizer=tokenizer,
                                          model=model)

            writer.writerow([i, row['application_number'], row['combined_string'], row['Ltext'], llama3_1_inference_result])
            print(f"{i}/{total_num} Row {i}: application_number: {row['application_number']}, combined_string: {row['combined_string']}, Ltext: {row['Ltext']}, prediction: {llama3_1_inference_result}")
            existing_app_numbers.add(app_number)  # 새로 추가된 번호는 추적
        else:
            print(f"Skipping Row {i}: application_number {row['application_number']} already exists.")

In [ ]:
from google.colab import files

# 파일 다운로드
files.download('test_output.csv')

In [ ]:
# CSV 파일 읽기
df = pd.read_csv('test_output.csv', header=None)

# 값이 동일한지 확인
equal_count = (df.iloc[:, 3] == df.iloc[:, 4]).sum()

# 전체 데이터에서 동일한 값이 차지하는 비율 계산
total_rows = len(df)
equal_percentage = (equal_count / total_rows) * 100

# 결과 출력
print(f"Meta-Llama-3.1-8B-Instruct 모델 예측 결과 정확도 계산('농업','임업', '어업') (Base 버전): {equal_percentage:.2f}%")
# Meta-Llama-3.1-8B-Instruct 모델 예측 결과 정확도 계산('농업','임업', '어업') (Base 버전): 42.68%

# Part 2 : 프롬프트 튜닝으로 Llama 3.1 성능 개선하기

In [ ]:
# 정답이 '임업'일 때
correct_label = '임업'

# 정답이 '임업'일 때 '임업'으로 예측한 비율
correct_forest_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '임업'일 때 '농업'으로 예측한 비율
correct_forest_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '임업'일 때 '어업'으로 예측한 비율
correct_forest_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '농업'일 때
correct_label = '농업'

# 정답이 '농업'일 때 '농업'으로 예측한 비율
correct_agriculture_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '농업'일 때 '임업'으로 예측한 비율
correct_agriculture_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '농업'일 때 '어업'으로 예측한 비율
correct_agriculture_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '어업'일 때
correct_label = '어업'

# 정답이 '어업'일 때 '어업'으로 예측한 비율
correct_fishery_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '어업'일 때 '임업'으로 예측한 비율
correct_fishery_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '어업'일 때 '농업'으로 예측한 비율
correct_fishery_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 결과 출력
print(f"정답이 '임업'일 때 '임업'으로 예측한 비율: {correct_forest_pred_forest:.2f}%")
print(f"정답이 '임업'일 때 '농업'으로 예측한 비율: {correct_forest_pred_agriculture:.2f}%")
print(f"정답이 '임업'일 때 '어업'으로 예측한 비율: {correct_forest_pred_fishery:.2f}%")

print(f"정답이 '농업'일 때 '농업'으로 예측한 비율: {correct_agriculture_pred_agriculture:.2f}%")
print(f"정답이 '농업'일 때 '임업'으로 예측한 비율: {correct_agriculture_pred_forest:.2f}%")
print(f"정답이 '농업'일 때 '어업'으로 예측한 비율: {correct_agriculture_pred_fishery:.2f}%")

print(f"정답이 '어업'일 때 '어업'으로 예측한 비율: {correct_fishery_pred_fishery:.2f}%")
print(f"정답이 '어업'일 때 '임업'으로 예측한 비율: {correct_fishery_pred_forest:.2f}%")
print(f"정답이 '어업'일 때 '농업'으로 예측한 비율: {correct_fishery_pred_agriculture:.2f}%")

In [ ]:
특허_카테고리_list = ['임업',
 '어업',
 '농업']

In [ ]:
특허_카테고리_list

In [ ]:
#기존 Prompt-Tuning전 프롬프트
system_prompt = f"너는 특허 카테고리를 분류하는 전문가야. \
아래 내용을 다음 특허 카테고리 중 하나로 분류해줘. 가능한 특허 카테고리 : {특허_카테고리_list}. \
최종 출력 결과는 다른말은 하지말고 분류한 카테고리만 출력해줘."
system_prompt

In [ ]:
#기존 프롬프트에서 Prompt-Tuning을 한 ㅍ롬프트
system_prompt_ver2 = f"너는 특허 카테고리를 분류하는 전문가야. \
아래 내용을 다음 특허 카테고리 중 하나로 분류해줘. 가능한 특허 카테고리 : {특허_카테고리_list}. \
'농업' 카테고리를 '임업' 카테고리를 분류하지 않도록 주의해. '임업'은 '삼림에서 주로 나무를 벌채하고 목재를 생산하는 산업'을 의미해. \
최종 출력 결과는 다른말은 하지말고 분류한 카테고리만 출력해줘."
system_prompt_ver2

In [ ]:
def generate_response(system_message, user_message, tokenizer, model, max_new_token):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=max_new_token,
        eos_token_id=terminators,
        do_sample=False,   # 항상 가장 확률이 높은 값으로 예측
        temperature=0.6,
        top_p=0.9
    )
    response = outputs[0][input_ids.shape[-1]:]

    return tokenizer.decode(response, skip_special_tokens=True)

In [ ]:
import csv

# 1. 기존 CSV 파일을 읽어서 application_number 목록을 추출
try:
    existing_df = pd.read_csv('test_output_ver2.csv', encoding='utf-8')
    existing_app_numbers = set(existing_df['application_number'].astype(str))
except FileNotFoundError:
    # 파일이 없을 경우 빈 set으로 초기화
    existing_app_numbers = set()

In [ ]:
total_num = len(filtered_df)

# 2. 새 데이터를 추가할 CSV 파일을 연다
with open('test_output_ver2.csv', mode='a', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, quoting=csv.QUOTE_MINIMAL)

    # 3. 데이터프레임 순회하면서 application_number가 중복되지 않으면 추가
    for i, (index, row) in enumerate(filtered_df.iterrows()):
        app_number = str(row['application_number'])  # 문자열로 변환하여 비교
        if app_number not in existing_app_numbers:
            llama3_1_inference_result = generate_response(system_message=system_prompt_ver2,
                                          user_message=row['combined_string'],
                                          tokenizer=tokenizer,
                                          model=model,
                                          max_new_token=512)

            writer.writerow([i, row['application_number'], row['combined_string'], row['Ltext'], llama3_1_inference_result])
            print(f"{i}/{total_num} Row {i}: application_number: {row['application_number']}, combined_string: {row['combined_string']}, Ltext: {row['Ltext']}, prediction: {llama3_1_inference_result}")
            existing_app_numbers.add(app_number)  # 새로 추가된 번호는 추적
        else:
            print(f"Skipping Row {i}: application_number {row['application_number']} already exists.")

In [ ]:
from google.colab import files

# 파일 다운로드
files.download('test_output_ver2.csv')

In [ ]:
# CSV 파일 읽기
df = pd.read_csv('test_output_ver2.csv', header=None)

# 값이 동일한지 확인
equal_count = (df.iloc[:, 3] == df.iloc[:, 4]).sum()

# 전체 데이터에서 동일한 값이 차지하는 비율 계산
total_rows = len(df)
equal_percentage = (equal_count / total_rows) * 100

# 결과 출력
print(f"Meta-Llama-3.1-8B-Instruct 모델 예측 결과 정확도 계산('농업','임업', '어업') (프롬프트 튜닝 버전 - Ver2): {equal_percentage:.2f}%")
# Meta-Llama-3.1-8B-Instruct 모델 예측 결과 정확도 계산('농업','임업', '어업') (Base 버전): 42.68%
# Meta-Llama-3.1-8B-Instruct 모델 예측 결과 정확도 계산('농업','임업', '어업') (프롬프트 튜닝 버전 - Ver2): 70.09%

In [ ]:
# 정답이 '임업'일 때
correct_label = '임업'

# 정답이 '임업'일 때 '임업'으로 예측한 비율
correct_forest_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '임업'일 때 '농업'으로 예측한 비율
correct_forest_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '임업'일 때 '어업'으로 예측한 비율
correct_forest_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '농업'일 때
correct_label = '농업'

# 정답이 '농업'일 때 '농업'으로 예측한 비율
correct_agriculture_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '농업'일 때 '임업'으로 예측한 비율
correct_agriculture_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '농업'일 때 '어업'으로 예측한 비율
correct_agriculture_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '어업'일 때
correct_label = '어업'

# 정답이 '어업'일 때 '어업'으로 예측한 비율
correct_fishery_pred_fishery = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '어업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '어업'일 때 '임업'으로 예측한 비율
correct_fishery_pred_forest = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '임업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 정답이 '어업'일 때 '농업'으로 예측한 비율
correct_fishery_pred_agriculture = ((df.iloc[:, 3] == correct_label) & (df.iloc[:, 4] == '농업')).sum() / (df.iloc[:, 3] == correct_label).sum() * 100

# 결과 출력
print(f"정답이 '임업'일 때 '임업'으로 예측한 비율: {correct_forest_pred_forest:.2f}%")
print(f"정답이 '임업'일 때 '농업'으로 예측한 비율: {correct_forest_pred_agriculture:.2f}%")
print(f"정답이 '임업'일 때 '어업'으로 예측한 비율: {correct_forest_pred_fishery:.2f}%")

print(f"정답이 '농업'일 때 '농업'으로 예측한 비율: {correct_agriculture_pred_agriculture:.2f}%")
print(f"정답이 '농업'일 때 '임업'으로 예측한 비율: {correct_agriculture_pred_forest:.2f}%")
print(f"정답이 '농업'일 때 '어업'으로 예측한 비율: {correct_agriculture_pred_fishery:.2f}%")

print(f"정답이 '어업'일 때 '어업'으로 예측한 비율: {correct_fishery_pred_fishery:.2f}%")
print(f"정답이 '어업'일 때 '임업'으로 예측한 비율: {correct_fishery_pred_forest:.2f}%")
print(f"정답이 '어업'일 때 '농업'으로 예측한 비율: {correct_fishery_pred_agriculture:.2f}%")
